In [86]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [87]:
file_name = r"C:\Users\admin\Documents\25.01.2023 £1,050.57 Co-ordSport.pdf"
r"C:\Users\admin\Documents\25.01.2023 £1,050.57 Co-ordSport.pdf"

'C:\\Users\\admin\\Documents\\25.01.2023 £1,050.57 Co-ordSport.pdf'

In [88]:
invoice_type = "Products"
# inputFolder = os.path.abspath('..\\Forge')

input_file = fr"C:\Users\admin\Documents\25.01.2023 £1,050.57 Co-ordSport.pdf"

In [89]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(135,677,176,762),
                  columns=[762],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
print(heading)

name = "Co-ordSport"
docnum = heading[0][0]
print(docnum)

date = heading[0][1]
#print(date)
date = str(datetime.strptime(date, "%d/%m/%Y"))
print(date)

            0
0    DS328902
1  25/01/2023
DS328902
2023-01-25 00:00:00


In [90]:
table2 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(133,107,195,218),
                  columns=[218],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading2 = table2[0]
print(heading2)

ordernum = heading2[0][1]
print(ordernum)

transfernum = None
print(transfernum)

                         0
0                 XX021325
1                   009858
2               RP/WEB 11:
3  UPS - UPS - all options
4                      CCD
009858
None


In [91]:
table3 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(215,90,462,800),
                  columns=[130,255,490,540,600,650,700,750,800],
                  pandas_options={'header': None},
                  encoding="windows-1254")
content=table3[0]

content

,0,1,2,3,4,5,6,7,8
0,**,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ustomer,Order Number: 009858,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,WEB-UPS-UK,UPS UK Standard Next Day (most mainland areas),GB,NaN,0.01,12.50,10.73,10.73
3,IVERED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ustomer,Order Number: T13956,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,MMBCC-F80-15CBE,Mishimoto Baffled Oil Catch Can fits BMW F8X M...,CN,8.709000e+09,0.00,249.65,187.24,187.24
6,NaN,NaN,202,NaN,NaN,NaN,NaN,NaN,NaN
7,ustomer,Order Number: T12880,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,184.70,138.53,277.06
9,NaN,NaN,2010,NaN,NaN,NaN,NaN,NaN,NaN


In [92]:

content = content[content[0].notnull()].iloc[:, 0:].reset_index(drop=True)  # Remove NaN
content = content.dropna(subset=[7]).reset_index(drop=True)  # Remove rows with NaN in column 7

content

,0,1,2,3,4,5,6,7,8
0,1,WEB-UPS-UK,UPS UK Standard Next Day (most mainland areas),GB,NaN,0.01,12.50,10.73,10.73
1,1,MMBCC-F80-15CBE,Mishimoto Baffled Oil Catch Can fits BMW F8X M...,CN,8.709000e+09,0.00,249.65,187.24,187.24
2,2,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,184.70,138.53,277.06
3,1,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,184.70,138.53,138.53
4,1,MMBCC-N20N26-12CB,Mishimoto Baffled Oil Catch Can CCV Side fits ...,CN,8.709000e+09,4.00,188.39,141.29,141.29
5,1,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,188.39,141.29,141.29


In [93]:
content.rename(columns={
    0: 'Qty',
    1: 'Part Number',
    2: 'Product Description',
    3: 'Country',
    4: 'Intrastat',
    5: 'Weight',
    6: 'RRP',
    7: 'Unit',
    8: 'Net Total'}, inplace=True)

display(content)

,Qty,Part Number,Product Description,Country,Intrastat,Weight,RRP,Unit,Net Total
0,1,WEB-UPS-UK,UPS UK Standard Next Day (most mainland areas),GB,NaN,0.01,12.50,10.73,10.73
1,1,MMBCC-F80-15CBE,Mishimoto Baffled Oil Catch Can fits BMW F8X M...,CN,8.709000e+09,0.00,249.65,187.24,187.24
2,2,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,184.70,138.53,277.06
3,1,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,184.70,138.53,138.53
4,1,MMBCC-N20N26-12CB,Mishimoto Baffled Oil Catch Can CCV Side fits ...,CN,8.709000e+09,4.00,188.39,141.29,141.29
5,1,MMBCC-N54-06CBE2,Mishimoto Baffled Oil Catch Can CCV Side BMW N...,CN,8.709000e+09,6.00,188.39,141.29,141.29


In [94]:
dict_content = content.to_dict(orient='records')
dict_content


line_items=[]
for item in dict_content:
    # print(item)
    
    partNum = item['Part Number']
    desc = item['Product Description']
    quantity = item['Qty']
    netTotal = item['Net Total']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": partNum,
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

WEB-UPS-UK
MMBCC-F80-15CBE
MMBCC-N54-06CBE2
MMBCC-N54-06CBE2
MMBCC-N20N26-12CB
MMBCC-N54-06CBE2
[{'line_type': 'inventory', 'sku': 'WEB-UPS-UK', 'name': 'UPS UK Standard Next Day (most mainland areas)', 'quantity': 1, 'net_total': 10.73, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-F80-15CBE', 'name': 'Mishimoto Baffled Oil Catch Can fits BMW F8X M3/M4 2015', 'quantity': 1, 'net_total': 187.24, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N54-06CBE2', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007', 'quantity': 2, 'net_total': 277.06, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N54-06CBE2', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007', 'quantity': 1, 'net_total': 138.53, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N20N26-12CB', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side fits BMW N20/', 'quantity': 1, 'net_total': 141.29, 'tax_type': 'INPUT2'}, {'line_type': 'inventory

In [95]:
table4 = read_pdf(input_file,
            pages="1",
            silent=True,
            guess=False,
            area=(440,660,531,800),
            columns=[750,800],
            pandas_options={'header': None},
            encoding='windows-1254')

total_content=table4[0]

#To remove "£"
total_content[[1]] = total_content[[1]].replace('[£, ]','', regex=True).astype('string')
#print(total_content)

total_content = total_content.dropna(subset=[0,1])  # Remove rows with NaN in column 0 & 1
total_content = total_content.dropna(subset=[0]).reset_index(drop=True)
display(total_content)

row_index = total_content.index[total_content[0] == "Grand Total:"].tolist()[0]

final_total = float(total_content[1][row_index])
display(final_total)

row_index2 = total_content.index[total_content[0] == "Carriage:"].tolist()[0]

Shipping = float(total_content[1][row_index2])
Shipping

,0,1
0,Goods:,885.41
1,Carriage:,10.73
2,VAT:,179.23
3,Grand Total:,1075.37


1075.37

10.73

In [96]:
#Adding shipping to the line_items
line_shipping = {"line_type":"shipping_expense",
            "sku": None,
            "name": "shipping",
            "Quantity": int(1),
            "net_total": float(Shipping),
            "tax_type": "INPUT2"}

line_items.append(line_shipping)
print(line_items)

[{'line_type': 'inventory', 'sku': 'WEB-UPS-UK', 'name': 'UPS UK Standard Next Day (most mainland areas)', 'quantity': 1, 'net_total': 10.73, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-F80-15CBE', 'name': 'Mishimoto Baffled Oil Catch Can fits BMW F8X M3/M4 2015', 'quantity': 1, 'net_total': 187.24, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N54-06CBE2', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007', 'quantity': 2, 'net_total': 277.06, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N54-06CBE2', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007', 'quantity': 1, 'net_total': 138.53, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N20N26-12CB', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side fits BMW N20/', 'quantity': 1, 'net_total': 141.29, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MMBCC-N54-06CBE2', 'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007', '

In [97]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Documents\\25.01.2023 £1,050.57 Co-ordSport.pdf',
 'Type': 'Products',
 'Name': 'Co-ordSport',
 'Date': '2023-01-25 00:00:00',
 'Reference No.': 'DS328902',
 'Order No.': '009858',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': 'WEB-UPS-UK',
   'name': 'UPS UK Standard Next Day (most mainland areas)',
   'quantity': 1,
   'net_total': 10.73,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMBCC-F80-15CBE',
   'name': 'Mishimoto Baffled Oil Catch Can fits BMW F8X M3/M4 2015',
   'quantity': 1,
   'net_total': 187.24,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMBCC-N54-06CBE2',
   'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007',
   'quantity': 2,
   'net_total': 277.06,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MMBCC-N54-06CBE2',
   'name': 'Mishimoto Baffled Oil Catch Can CCV Side BMW N54 2007',
   'quantity': 1,
   'net